In [13]:
# Setup & Drive Mounting
from google.colab import drive
import os
import glob
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset

# Mount Google Drive
drive.mount('/content/drive')

# Configuration
CONFIG = {}
base_path = "/content/drive/MyDrive/NLP Project/subtask1/"

CONFIG["TRAIN_DIR"] = os.path.join(base_path, "train/")
CONFIG["DEV_DIR"]   = os.path.join(base_path, "dev/")
CONFIG["SAVE_DIR"]  = os.path.join(base_path, "xlm_roberta/models/")
CONFIG["PRED_DIR"]  = os.path.join(base_path, "xlm_roberta/predictions/")

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
os.makedirs(CONFIG["PRED_DIR"], exist_ok=True)

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sat Jan 10 20:06:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             52W /  400W |    5935MiB /  40960MiB |      0%      Default |
|          

In [14]:
# Data Loading Helper Function
def load_data_from_folder(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    df_list = []

    print(f"Found {len(all_files)} files in {folder_path}")

    for filename in all_files:
        try:
            df = pd.read_csv(filename)
            lang_code = os.path.basename(filename).split('.')[0]
            df['lang'] = lang_code
            df_list.append(df)
            print(f" - Loaded: {os.path.basename(filename)} ({len(df)} rows)")
        except Exception as e:
            print(f" - Error loading {filename}: {e}")

    if not df_list:
        raise ValueError(f"No CSV files found in {folder_path}")

    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

In [15]:
# Load, Clean and Split Data
print("\n--- Loading Labeled Training Data ---")
# Load all training files into one big table
full_train_df = load_data_from_folder(CONFIG["TRAIN_DIR"])

# Drop rows where text or polarization is missing
full_train_df.dropna(subset=['text', 'polarization'], inplace=True)
full_train_df['text'] = full_train_df['text'].astype(str)
full_train_df['polarization'] = full_train_df['polarization'].astype(int)

print(f"Total Labeled Samples: {len(full_train_df)}")

print("\n--- Splitting Data (80% Train / 20% Internal Test) ---")

train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.20, # 20% for evaluation
    random_state=42,
    stratify=full_train_df['polarization'] # Keeps the ratio of 0s and 1s balanced
)

print(f"   Training Set: {len(train_df)} samples")
print(f"   Internal Test Set (val_df): {len(val_df)} samples")


# Load blind test data
print("\n--- Loading Blind Dev Data ---")
test_df = load_data_from_folder(CONFIG["DEV_DIR"])
test_df['text'] = test_df['text'].astype(str)

print(f"Blind Test Samples: {len(test_df)}")


--- Loading Labeled Training Data ---
Found 13 files in /content/drive/MyDrive/NLP Project/subtask1/train/
 - Loaded: tur.csv (2364 rows)
 - Loaded: zho.csv (4280 rows)
 - Loaded: urd.csv (2849 rows)
 - Loaded: spa.csv (3305 rows)
 - Loaded: nep.csv (2005 rows)
 - Loaded: ita.csv (3334 rows)
 - Loaded: hin.csv (2744 rows)
 - Loaded: fas.csv (3295 rows)
 - Loaded: hau.csv (3651 rows)
 - Loaded: eng.csv (2676 rows)
 - Loaded: deu.csv (3180 rows)
 - Loaded: arb.csv (3380 rows)
 - Loaded: amh.csv (3332 rows)
Total Labeled Samples: 40395

--- Splitting Data (80% Train / 20% Internal Test) ---
   Training Set: 32316 samples
   Internal Test Set (val_df): 8079 samples

--- Loading Blind Dev Data ---
Found 13 files in /content/drive/MyDrive/NLP Project/subtask1/dev/
 - Loaded: zho.csv (214 rows)
 - Loaded: urd.csv (142 rows)
 - Loaded: tur.csv (115 rows)
 - Loaded: spa.csv (165 rows)
 - Loaded: nep.csv (100 rows)
 - Loaded: ita.csv (166 rows)
 - Loaded: hin.csv (137 rows)
 - Loaded: hau.csv (

In [27]:
# Tokenization & Dataset Preparation
from transformers import AutoTokenizer
from datasets import Dataset

print("\n--- Loading Tokenizer ---")
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Helper function to tokenize text
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("\n--- Converting Dataframes to Datasets ---")

# Prepare Training Data
train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(tokenize_function, batched=True)

# Prepare Validation Data (Internal Test with Labels)
val_dataset = Dataset.from_pandas(val_df)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Prepare Blind Test Data (No Labels)
# If 'polarization' column exists but is empty/NaN, drop it
if 'polarization' in test_df.columns:
    test_df = test_df.drop(columns=['polarization'])

test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Formatting for Pytorch
print("\n--- Formatting Columns for Model ---")

# Rename 'polarization' to 'labels' because Hugging Face expects that name
train_dataset = train_dataset.rename_column("polarization", "labels")
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

val_dataset = val_dataset.rename_column("polarization", "labels")
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# For Blind Test, we only need inputs
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

print(f" Tokenization Complete!")
print(f" Train Set: {len(train_dataset)} (Ready for training)")
print(f" Val Set: {len(val_dataset)} (Ready for F1 evaluation)")
print(f" Test Set: {len(test_dataset)} (Ready for prediction)")


--- Loading Tokenizer ---

--- Converting Dataframes to Datasets ---


Map:   0%|          | 0/32316 [00:00<?, ? examples/s]

Map:   0%|          | 0/8079 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]


--- Formatting Columns for Model ---
 Tokenization Complete!
 Train Set: 32316 (Ready for training)
 Val Set: 8079 (Ready for F1 evaluation)
 Test Set: 2012 (Ready for prediction)


In [17]:
import torch
import numpy as np
from transformers import TrainerCallback

# Model Training & Custom Logging
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Metrics Function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate Probabilities for the 'Polarized' class (Index 1)
    # Convert logits to probabilities using Softmax
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

    # Standard Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)

    # Custom Stats
    avg_prob = np.mean(probs)
    max_prob = np.max(probs)
    min_prob = np.min(probs)
    perc_pred_pos = np.mean(predictions) * 100  # % predictions > 0.5
    perc_true_pos = np.mean(labels) * 100       # % true positives in labels

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'avg_prob': avg_prob,
        'max_prob': max_prob,
        'min_prob': min_prob,
        'perc_pred_pos': perc_pred_pos,
        'perc_true_pos': perc_true_pos
    }

# Custom Callback for Text Logging
class TextLoggerCallback(TrainerCallback):
    def __init__(self, filepath):
        self.filepath = filepath
        # Create/Clear the file
        with open(self.filepath, 'w') as f:
            f.write("Training Results Log\n")
            f.write("====================\n\n")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        epoch = state.epoch

        # Get the latest training loss (if available in history)
        train_loss = "N/A"
        if state.log_history:
            for log in reversed(state.log_history):
                if 'loss' in log:
                    train_loss = f"{log['loss']:.4f}"
                    break

        # Extract metrics
        val_loss = metrics.get('eval_loss', 0.0)
        acc = metrics.get('eval_accuracy', 0.0)
        f1 = metrics.get('eval_f1', 0.0)
        prec = metrics.get('eval_precision', 0.0)
        rec = metrics.get('eval_recall', 0.0)

        # Extract Custom Stats
        avg_prob = metrics.get('eval_avg_prob', 0.0)
        max_prob = metrics.get('eval_max_prob', 0.0)
        min_prob = metrics.get('eval_min_prob', 0.0)
        perc_pred = metrics.get('eval_perc_pred_pos', 0.0)
        perc_true = metrics.get('eval_perc_true_pos', 0.0)

        log_str = (
            f"Epoch {int(epoch)}/{args.num_train_epochs}\n"
            f"----------------------------------------\n"
            f"Train loss: {train_loss:<15}\n"
            f"\n"
            f"  Prediction stats (threshold=0.5):\n"
            f"    Avg probability: {avg_prob:.4f}\n"
            f"    Max probability: {max_prob:.4f}\n"
            f"    Min probability: {min_prob:.4f}\n"
            f"    % predictions > 0.5: {perc_pred:.2f}%\n"
            f"    % true positives in labels: {perc_true:.2f}%\n"
            f"Val loss: {val_loss:.4f}\n"
            f"Val Accuracy: {acc:.4f}\n"
            f"Val Binary F1: {f1:.4f}\n"
            f"  Precision: {prec:.4f}\n"
            f"  Recall: {rec:.4f}\n"
            f"\n"
        )

        print(log_str)

        with open(self.filepath, 'a') as f:
            f.write(log_str)

# Define output file path
log_file_path = os.path.join(CONFIG["SAVE_DIR"], "subtask1_training_log.txt")

# Define training arguments
training_args = TrainingArguments(
    output_dir=CONFIG["SAVE_DIR"],
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[TextLoggerCallback(log_file_path)]
)

print(f"\nStarting Training... Logs will be saved to: {log_file_path}")
trainer.train()

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-254803279.py:119: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



Starting Training... Logs will be saved to: /content/drive/MyDrive/NLP Project/subtask1/xlm_roberta/models/subtask1_training_log.txt


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Avg Prob,Max Prob,Min Prob,Perc Pred Pos,Perc True Pos
1,0.526700,0.437768,0.786112,0.782915,0.829164,0.741552,0.474941,0.984382,0.004230,46.515658,52.011388
2,0.405000,0.437424,0.799728,0.799951,0.832476,0.769871,0.477858,0.994445,0.001926,48.100012,52.011388
3,0.325600,0.477310,0.798490,0.796194,0.839937,0.756782,0.472188,0.997672,0.000843,46.862235,52.011388
4,0.254300,0.520224,0.798242,0.805536,0.807656,0.803427,0.517092,0.998958,0.000707,51.739077,52.011388
5,0.201200,0.598618,0.797128,0.800875,0.818069,0.784388,0.497374,0.999344,0.000341,49.870033,52.011388


Epoch 1/5
----------------------------------------
Train loss: 0.5267         

  Prediction stats (threshold=0.5):
    Avg probability: 0.4749
    Max probability: 0.9844
    Min probability: 0.0042
    % predictions > 0.5: 46.52%
    % true positives in labels: 52.01%
Val loss: 0.4378
Val Accuracy: 0.7861
Val Binary F1: 0.7829
  Precision: 0.8292
  Recall: 0.7416


Epoch 2/5
----------------------------------------
Train loss: 0.4050         

  Prediction stats (threshold=0.5):
    Avg probability: 0.4779
    Max probability: 0.9944
    Min probability: 0.0019
    % predictions > 0.5: 48.10%
    % true positives in labels: 52.01%
Val loss: 0.4374
Val Accuracy: 0.7997
Val Binary F1: 0.8000
  Precision: 0.8325
  Recall: 0.7699


Epoch 3/5
----------------------------------------
Train loss: 0.3256         

  Prediction stats (threshold=0.5):
    Avg probability: 0.4722
    Max probability: 0.9977
    Min probability: 0.0008
    % predictions > 0.5: 46.86%
    % true positives in labe

TrainOutput(global_step=5050, training_loss=0.34254008378132733, metrics={'train_runtime': 943.0171, 'train_samples_per_second': 171.344, 'train_steps_per_second': 5.355, 'total_flos': 1.06283710812672e+16, 'train_loss': 0.34254008378132733, 'epoch': 5.0})

In [20]:
# Save Model & Final Log Entry
from sklearn.metrics import classification_report, confusion_matrix

# Save the Best Model
final_path = os.path.join(CONFIG["SAVE_DIR"], "best_model")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Best model saved to: {final_path}")

# Run Final Prediction on Internal Test Set
print("\n--- Running Final Evaluation ---")
val_output = trainer.predict(val_dataset)

# Calculate Final Metrics
val_preds = np.argmax(val_output.predictions, axis=-1)
val_labels = val_output.label_ids
probs = torch.nn.functional.softmax(torch.tensor(val_output.predictions), dim=-1)[:, 1].numpy()

# Calculate stats for the log
precision, recall, f1, _ = precision_recall_fscore_support(val_labels, val_preds, average='binary')
acc = accuracy_score(val_labels, val_preds)
avg_prob = np.mean(probs)
max_prob = np.max(probs)
min_prob = np.min(probs)
perc_pred_pos = np.mean(val_preds) * 100
perc_true_pos = np.mean(val_labels) * 100

final_log_str = (
    f"\n"
    f"============================================================\n"
    f"FINAL TEST EVALUATION - {MODEL_NAME}\n"
    f"============================================================\n"
    f"  Prediction stats (threshold=0.5):\n"
    f"    Avg probability: {avg_prob:.4f}\n"
    f"    Max probability: {max_prob:.4f}\n"
    f"    Min probability: {min_prob:.4f}\n"
    f"    % predictions > 0.5: {perc_pred_pos:.2f}%\n"
    f"    % true positives in labels: {perc_true_pos:.2f}%\n"
    f"Test Loss: {val_output.metrics.get('test_loss', 'N/A')}\n"
    f"Test Binary F1: {f1:.4f}\n"
    f"Test Accuracy: {acc:.4f}\n"
    f"\n"
)

print(final_log_str)

log_file_path = os.path.join(CONFIG["SAVE_DIR"], "subtask1_training_log.txt")
with open(log_file_path, 'a') as f:
    f.write(final_log_str)

print(f"Final stats appended to: {log_file_path}")

print("\nDetailed Classification Report:")
print(classification_report(val_labels, val_preds, target_names=['Neutral', 'Polarized'], digits=4))

Best model saved to: /content/drive/MyDrive/NLP Project/subtask1/xlm_roberta/models/best_model

--- Running Final Evaluation ---



FINAL TEST EVALUATION - xlm-roberta-base
  Prediction stats (threshold=0.5):
    Avg probability: 0.5171
    Max probability: 0.9990
    Min probability: 0.0007
    % predictions > 0.5: 51.74%
    % true positives in labels: 52.01%
Test Loss: 0.5202243328094482
Test Binary F1: 0.8055
Test Accuracy: 0.7982


Final stats appended to: /content/drive/MyDrive/NLP Project/subtask1/xlm_roberta/models/subtask1_training_log.txt

Detailed Classification Report:
              precision    recall  f1-score   support

     Neutral     0.7882    0.7926    0.7904      3877
   Polarized     0.8077    0.8034    0.8055      4202

    accuracy                         0.7982      8079
   macro avg     0.7979    0.7980    0.7980      8079
weighted avg     0.7983    0.7982    0.7983      8079



In [21]:
# Generate Predictions on Blind Test Data
print("\n--- Predicting on Blind Test Data ---")

# Use the trainer to predict on the pre-tokenized 'test_dataset'
predictions_output = trainer.predict(test_dataset)
test_preds = np.argmax(predictions_output.predictions, axis=-1)

# Attach predictions back to the original dataframe
submission_df = test_df.copy()
submission_df['polarization'] = test_preds

print(f"Generated {len(submission_df)} predictions.")


print("\n--- Saving Individual Language Files ---")

# Use the 'lang' column to split them back up
unique_langs = submission_df['lang'].unique()

for lang in unique_langs:
    # Filter for just this language
    lang_df = submission_df[submission_df['lang'] == lang]

    final_output = lang_df[['id', 'polarization']]
    save_name = f"pred_{lang}.csv"
    save_path = os.path.join(CONFIG["PRED_DIR"], save_name)

    final_output.to_csv(save_path, index=False)
    print(f"   Saved: {save_name} ({len(final_output)} rows)")

print("\n--- Saving Combined CSV ---")
combined_path = os.path.join(CONFIG["PRED_DIR"], "combined.csv")
submission_df[['id', 'polarization']].to_csv(combined_path, index=False)
print(f"Combined file saved to: {combined_path}")


--- Predicting on Blind Test Data ---


Generated 2012 predictions.

--- Saving Individual Language Files ---
   Saved: pred_zho.csv (214 rows)
   Saved: pred_urd.csv (142 rows)
   Saved: pred_tur.csv (115 rows)
   Saved: pred_spa.csv (165 rows)
   Saved: pred_nep.csv (100 rows)
   Saved: pred_ita.csv (166 rows)
   Saved: pred_hin.csv (137 rows)
   Saved: pred_hau.csv (182 rows)
   Saved: pred_fas.csv (164 rows)
   Saved: pred_deu.csv (159 rows)
   Saved: pred_arb.csv (169 rows)
   Saved: pred_amh.csv (166 rows)
   Saved: pred_eng.csv (133 rows)

--- Saving Combined CSV ---
Combined file saved to: /content/drive/MyDrive/NLP Project/subtask1/xlm_roberta/predictions/combined.csv
